# 🎭 Charla en Colab (pipeline completo con GPU)

Corre TODO el generador en Colab: guion (Claude/Gemini), voz clonada **Chatterbox con GPU** (segundos por línea en vez de minutos), chromakey y render ffmpeg. Tú usas la interfaz Gradio desde el navegador.

El código se clona del repo en cada sesión — para actualizar Colab basta hacer `git push` y re-ejecutar la celda 1. Abre siempre la versión más reciente de este cuaderno desde GitHub:
`https://colab.research.google.com/github/sfsusuario/storytelling-conversation/blob/main/colab/charla_colab.ipynb`

**Preparación (una sola vez):** en tu Drive debe existir `MyDrive/charla/` con los archivos que NO viajan por git:
- `characters/` y `background.mp4` — los assets de video
- `voices_preview/reales/` — los clips de referencia de las voces
- `.env` con tu `ANTHROPIC_API_KEY` o `GOOGLE_API_KEY`

**Runtime:** GPU (T4). `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.

In [ ]:
#@title ⏳ 1 - Preparar el entorno (Drive + repo + dependencias, ~3-5 min)
import os, shutil, sys
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/charla'  #@param {type:'string'}
REPO_URL = 'https://github.com/sfsusuario/storytelling-conversation.git'  #@param {type:'string'}
WORK_DIR = '/content/charla'

assert os.path.isdir(DRIVE_DIR), f'No existe {DRIVE_DIR}: crea la carpeta en tu Drive primero.'

# Código: clon fresco del repo. Medios y .env: desde Drive a disco local.
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
!git clone --depth 1 {REPO_URL} {WORK_DIR}
assert os.path.isfile(f'{WORK_DIR}/pyproject.toml'), 'El clone falló: revisa REPO_URL.'
for media in ('characters', 'voices_preview'):
    src = f'{DRIVE_DIR}/{media}'
    assert os.path.isdir(src), f'Falta {src}: arrastra esa carpeta a tu Drive.'
    shutil.copytree(src, f'{WORK_DIR}/{media}', dirs_exist_ok=True)
assert os.path.isfile(f'{DRIVE_DIR}/background.mp4'), 'Falta background.mp4 en Drive.'
shutil.copy(f'{DRIVE_DIR}/background.mp4', WORK_DIR)
assert os.path.isfile(f'{DRIVE_DIR}/.env'), f'Falta {DRIVE_DIR}/.env con tus claves de API.'
shutil.copy(f'{DRIVE_DIR}/.env', WORK_DIR)
os.chdir(WORK_DIR)

# El modelo Chatterbox (~3 GB) se cachea en Drive para no bajarlo cada sesión
os.environ['HF_HOME'] = f'{DRIVE_DIR}/.hf_cache'

# En Colab los workers corren en el propio runtime (no hay venvs de Windows)
os.environ['CHARLA_CHATTERBOX_PYTHON'] = sys.executable
os.environ['CHARLA_TTS'] = 'chatterbox'

!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q -e .[ui] chatterbox-tts
# Combo verificado con chatterbox 0.1.7: transformers 5.2.0 + torch 2.6.
# El torchvision preinstalado de Colab (compilado para otro torch) rompe los
# imports de transformers y no lo usamos: fuera.
!pip install -q "transformers==5.2.0"
!pip uninstall -y -q torchvision
# chatterbox degrada starlette/fastapi y rompe el render de la UI de Gradio;
# reinstalar gradio realinea el stack web...
!pip install -q --force-reinstall "gradio==6.20.0"
# ...pero ese force-reinstall sube numpy/pandas de más: re-anclar las
# versiones que chatterbox y el propio Colab necesitan.
!pip install -q "numpy==1.26.4" "pandas==2.2.2"

# Verificación final del worker de voz
!python -c "from chatterbox.mtl_tts import ChatterboxMultilingualTTS; print('chatterbox OK')"

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO (activa el runtime T4)')
print('Listo. Ejecuta la celda 2 (interfaz) o la 3 (CLI).')

In [ ]:
#@title 🎬 2 - Interfaz web (Gradio)
#@markdown Abre el enlace `https://....gradio.live` que aparece abajo y usa la UI normal.
#@markdown El motor de voz por defecto en Colab es **chatterbox** (con GPU).
#@markdown Los videos quedan en `output/` local: guárdalos en Drive con la celda 4.
import sys, importlib, os

# Import robusto: independiente del estado del kernel y del editable install
os.chdir('/content/charla')
for m in [k for k in list(sys.modules) if k == 'charla' or k.startswith('charla.')]:
    del sys.modules[m]
if '/content/charla/src' not in sys.path:
    sys.path.insert(0, '/content/charla/src')
importlib.invalidate_caches()

from charla.ui import build_app

build_app().launch(share=True, debug=True)

In [ ]:
#@title ⌨️ 3 - (Alternativa) Generar por CLI
tema = 'los pulpos tienen tres corazones y sangre azul'  #@param {type:'string'}

!python -m charla.cli "{tema}"

In [ ]:
#@title 💾 4 - Guardar los videos en Drive
import glob, os, shutil

dest = f'{DRIVE_DIR}/output'
os.makedirs(dest, exist_ok=True)
for video in glob.glob('output/*/final.mp4'):
    name = os.path.basename(os.path.dirname(video))
    shutil.copy(video, f'{dest}/{name}.mp4')
    social = os.path.join(os.path.dirname(video), 'social.txt')
    if os.path.exists(social):
        shutil.copy(social, f'{dest}/{name}.social.txt')
    print(f'guardado: {dest}/{name}.mp4')